# AlphaEarth Embeddings → Outlier Scores Pipeline

Scores WorldCereal samples using **64-dim AlphaEarth embeddings** fetched.


## Outlier Scoring

| # | Section | Description |
|---|---------|-------------|
| 1 | **Parameters** | Paths & knobs — point `EMBEDDINGS_DB_PATH` at the COG DuckDB |
| 2 | **Inspect DuckDB** | Schema, deduplication |
| 2b | **Deduplicate** | Keep one row per `sample_id` |
| 2c | **Regional coverage** | Cross-check against global parquet |
| 3 | **Load region parquet** | Clean anomaly cols, check overlap |
| 4 | **Prepare embeddings** | Rename `A00`→`embedding_0`, join metadata |
| 5 | **Class mappings** | Local JSON or SharePoint |
| 6–7 | **LC10 / CTY24 scoring** | `run_pipeline(embeddings_df=...)` |
| 8–9 | **Merge & write back** | Save scored parquet |

> **COG encoding:** `int8`, nodata=`-128`, scale=`1/127` → float32 in `[-1, 1]`.  
> Band names `A00`–`A63` in the COG match the DuckDB column names exactly.


## ⚠️ Updated detector — read before running

The scoring logic changed. This notebook has been updated to match; the notes
below explain what is different and which knobs you may want to change.

**A flag now requires two conditions, not one.**
Every within-slice score is percentile-normalised *per slice*, so on its own it
cannot tell *the most unusual point of a clean slice* from *a mislabelled point*
— the top ~2 % of every slice cleared the old escalation thresholds whether or
not the slice held a single real error. And the old `median + k·MAD` gate was a
knife-edge: on synthetic data it flagged **0 %** at both 2 % and 30 % true
contamination, but 9 % at 10 %, because a contaminated slice inflates its own
median and MAD. So a sample must now be:

1. **locally unusual** — `threshold_mode="stable_mad"` keeps the slice median as
   the local reference but takes the *dispersion* from a cross-slice null, which
   a single slice cannot inflate; and
2. **absolutely unusual** — `abs_z >= abs_z_k` robust sigma against a null pooled
   across many slices of the same class, built from one summary statistic per
   slice so a dirty slice cannot calibrate its own errors away.

Set `require_absolute=False` to reproduce the old relative-only behaviour.

**`group_cols` now defaults to `["ref_id"]`.**
Slices are scored per source dataset, so one dataset's labelling convention
cannot contaminate another's reference cloud. **Pass `group_cols=[]` to pool
across datasets** — the previous behaviour of these notebooks. Pooling lets a
dataset digitised against a different legend either mask itself (if it is large)
or get flagged wholesale (if it is small). The *context* used for the kNN-purity
and alt-class-margin signals stays geographic either way; override it with
`context_group_cols` only if you know you want per-dataset context.

**New flag values.** `*_anomaly_flag` can now also be:

| value | meaning |
|---|---|
| `unscored` | slice too small to score — **not** the same as `normal` |
| `unscorable` | embedding failed the quality gate (zero-norm, non-finite, duplicate id); see `quality_reason` |
| `unmapped` | `ewoc_code` absent from the legend — a coverage gap |
| `skipped` | held out by `skip_classes` |

These are terminal states recording that the detector *could not form an
opinion*. Any code that ranks flags must not fold them into `normal`; the helper
cells below use `_NON_JUDGED_FLAGS` for this. They are also never dropped by the
`experiments/scenarios.py` drop sets — "we did not look" must not remove a
training sample.

**Escalation is stricter.** `suspect` / `candidate` need agreement across signals
that measure different things (absolute distance, kNN label purity, alt-class
margin, within-slice rank). A point whose neighbours overwhelmingly share its
label is capped at `flagged` (`purity_veto`), and where a context contains a
single label — no alternative class to have been confused with — escalation is
capped at `flagged` too.

**Evidence is kept.** The review parquets now carry `abs_z`, `cosine_distance`,
`knn_distance`, `neighbourhood_offset`, `knn_same_label_frac_ctx`,
`alt_margin_ctx`, `escalation_votes`, `weak_support`, `purity_veto`,
`corroborated` and `quality_reason`, so a flag can be audited against a basemap
instead of taken on trust.

**Update — heavy slice contamination.** Defaults changed again after measuring
the 30–45 % regime end-to-end. The blocker was the *scale*, not the centroid:
the cross-slice null took the median of per-slice **MADs**, and a MAD is
widened by the very right-side errors it measures against — so when every slice
of a class carried 30–45 % errors the null inflated with them and nothing
cleared the gate. `null_scale_estimator="left_tail"` measures the spread as
`median − q25`, from the clean left half only.

| slice contamination | recall before | recall now |
|---|---|---|
| 20 % | 0.992 | 0.994 |
| 30 % | 0.798 | 0.985 |
| 40 % | 0.140 | 0.902 |
| 45 % | 0.084 | 0.256 |

…at a clean-data false-positive rate of 1.11 %, slightly **below** the previous
1.17 %. Precision stays above 0.98 throughout. New defaults: `mad_k=3.3`,
`abs_z_k=3.3`, `centroid_trim=0.45` (the trim must be ≥ the worst contamination
you expect), `null_scale_estimator="left_tail"`. Pass
`null_scale_estimator="mad"` for the legacy ablation.

**Expect different counts from previous runs.** Re-run end-to-end rather than
mixing old and new outputs in the same merge.


---
## Outlier Scoring on AlphaEarth Embeddings

The cells below use the DuckDB produced in Part 1 (`alphaearth_cog_embeddings.duckdb`) to run the `run_pipeline` outlier detection — the same pipeline as the standard Presto-based workflow, but using AlphaEarth 64-dim embeddings fetched directly from Source Cooperative COGs.

> **If you already have the COG DuckDB** from a previous run, you can skip Part 1 and jump straight here. Just set `COG_EMBEDDINGS_DB` in the parameters cell below to point to it.

In [1]:

# ==============================================================
# MERGE AVAILABLE CHUNKS → COG_EMBEDDINGS_PARQUET
# Run this FIRST — picks up all chunk_*.parquet / chunk_h_*.parquet
# that exist right now (safe to re-run at any time).
# ==============================================================
import pandas as pd
from pathlib import Path

_DATA                  = Path("/path/to/TestFolder/wc_outliers/data_for_outlier")
CHUNKS_DIR             = _DATA / "alphaearth_cog_chunks_eastern_africa"
HEAVY_DIR              = CHUNKS_DIR / "heavy"
COG_EMBEDDINGS_PARQUET = _DATA / "alphaearth_cog_embeddings_eastern_africa.parquet"

regular_chunks = sorted(CHUNKS_DIR.glob("chunk_*.parquet"))
heavy_chunks   = sorted(HEAVY_DIR.glob("chunk_h_*.parquet")) if HEAVY_DIR.exists() else []
all_chunks     = regular_chunks + heavy_chunks

assert all_chunks, f"No chunk files found in {CHUNKS_DIR} — run the fetch cell first!"
print(f"Regular chunks : {len(regular_chunks)}")
print(f"Heavy chunks   : {len(heavy_chunks)}")
print(f"Total          : {len(all_chunks)}  →  merging …")

_df = pd.concat([pd.read_parquet(str(f)) for f in all_chunks], ignore_index=True)
_df.drop_duplicates(subset="sample_id", inplace=True)
_df.to_parquet(str(COG_EMBEDDINGS_PARQUET), index=False)

_n_valid = int(_df["valid"].sum())
print(f"\n✓  {COG_EMBEDDINGS_PARQUET.name}")
print(f"   Rows  : {len(_df):,}")
print(f"   Valid : {_n_valid:,}  ({100*_n_valid/max(len(_df),1):.1f}%)")
print(f"   Year  : {_df['year'].value_counts().sort_index().to_dict()}")
del _df   # free memory — loaded properly in cell below


Regular chunks : 0
Heavy chunks   : 99
Total          : 99  →  merging …

✓  alphaearth_cog_embeddings_eastern_africa.parquet
   Rows  : 115,531
   Valid : 115,107  (99.6%)
   Year  : {2017: 2533, 2018: 18248, 2019: 10812, 2020: 10725, 2021: 29186, 2022: 11512, 2023: 14471, 2024: 8997, 2025: 9047}


## 1) Parameters

Edit the paths and knobs below before running any other cell.

In [2]:

from __future__ import annotations

import gc
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from loguru import logger
from tqdm.auto import tqdm

from EBA_detector.anomaly import run_pipeline

# ============================================================
# REGION TAG  — change this to run on a different region file
# ============================================================
REGION = "Eastern_Africa"  # ← change this to run on a different region file

# ============================================================
# PATHS
# ============================================================
_DATA = Path("/path/to/TestFolder/wc_outliers/data_for_outlier")

COG_EMBEDDINGS_PARQUET = _DATA / "alphaearth_cog_embeddings_eastern_africa.parquet"   # ← from merge cell
INPUT_PARQUET          = _DATA / f"{REGION}.parquet"

_OUT_DIR = _DATA / f"alphaearth_scores_{REGION}"
_OUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PARQUET     = _DATA / f"{REGION}_with_alphaearth_scores.parquet"
MERGED_SCORES_PATH = _OUT_DIR / f"merged_LC10_CTY24_{REGION}.parquet"

OUT_LC10_DIR  = _OUT_DIR / "LC10"
OUT_CTY24_DIR = _OUT_DIR / "CTY24"
OUT_LC10_DIR.mkdir(parents=True, exist_ok=True)
OUT_CTY24_DIR.mkdir(parents=True, exist_ok=True)

LC10_SAMPLES_PATH  = str(OUT_LC10_DIR / f"LC10_samples_{REGION}.parquet")
LC10_SUMMARY_PATH  = str(OUT_LC10_DIR / f"LC10_summary_{REGION}.parquet")
CTY24_SAMPLES_PATH = str(OUT_CTY24_DIR / f"CTY24_samples_{REGION}.parquet")
CTY24_SUMMARY_PATH = str(OUT_CTY24_DIR / f"CTY24_summary_{REGION}.parquet")

# ============================================================
# ANOMALY COLUMNS
# ============================================================
ANOMALY_COLS = [
    "CTY24_confidence_nonoutlier",
    "CTY24_anomaly_flag",
    "outlier_CTY24_cls",
    "LC10_confidence_nonoutlier",
    "LC10_anomaly_flag",
    "outlier_LC10_cls",
]

# ============================================================
# SCORING KNOBS
# ============================================================
MODEL_HASH_PLACEHOLDER = "alphaearth_v1"
SKIP_CLASSES = ["ignore", "no_crop", "trees"]

# LC10_H3_LEVELS    = [2, 3]
LC10_H3_LEVELS    = [1, 2]
LC10_MAX_SLICE    = 10_000
LC10_MIN_SLICE    = 200
LC10_MAX_MERGE    = 16

CTY24_H3_LEVELS   = [1, 2, 3]
CTY24_MAX_SLICE   = 5_000
CTY24_MIN_SLICE   = 100
CTY24_MAX_MERGE   = 8

MAD_K             = 3.3   # stable_mad uses the cross-slice sigma; 3.0 is the new default
NORM_PERCENTILES  = (2.0, 98.0)
CENTROID_MODE     = "trimmed"
CENTROID_TRIM     = 0.45  # trim >= the largest contamination you expect in a slice
# ============================================================
# CLASS MAPPINGS
# ============================================================
import glob as _glob
_cm_candidates = _glob.glob(
    "/path/to/.conda/envs/*/lib/python*/site-packages/"
    "worldcereal/data/croptype_mappings/class_mappings.json"
) + _glob.glob(
    "/path/to/envs/*/lib/python*/site-packages/"
    "worldcereal/data/croptype_mappings/class_mappings.json"
) + _glob.glob(
    "/opt/conda/envs/*/lib/python*/site-packages/"
    "worldcereal/data/croptype_mappings/class_mappings.json"
)
LOCAL_CLASS_MAPPINGS_JSON = Path(_cm_candidates[0]) if _cm_candidates else None

SHAREPOINT_ENV_CANDIDATES = [
    Path("~/.sharepointenv"),
    Path("/path/to/TestFolder/.sharepointenv"),
]

# ============================================================
# SANITY CHECKS
# ============================================================
assert COG_EMBEDDINGS_PARQUET.exists(), f"Not found: {COG_EMBEDDINGS_PARQUET}  (run merge cell first)"
assert INPUT_PARQUET.exists(),          f"Not found: {INPUT_PARQUET}"

print(f"Region            : {REGION}")
print(f"Embeddings parquet: {COG_EMBEDDINGS_PARQUET}  ({COG_EMBEDDINGS_PARQUET.stat().st_size/1e6:.0f} MB)")
print(f"Input parquet     : {INPUT_PARQUET}")
print(f"Output parquet    : {OUTPUT_PARQUET}")
print(f"Merged scores     : {MERGED_SCORES_PATH}")
print(f"Local class map   : {LOCAL_CLASS_MAPPINGS_JSON}")


Region            : Eastern_Africa
Embeddings parquet: /path/to/TestFolder/wc_outliers/data_for_outlier/alphaearth_cog_embeddings_eastern_africa.parquet  (10 MB)
Input parquet     : /path/to/TestFolder/wc_outliers/data_for_outlier/Eastern_Africa.parquet
Output parquet    : /path/to/TestFolder/wc_outliers/data_for_outlier/Eastern_Africa_with_alphaearth_scores.parquet
Merged scores     : /path/to/TestFolder/wc_outliers/data_for_outlier/alphaearth_scores_Eastern_Africa/merged_LC10_CTY24_Eastern_Africa.parquet
Local class map   : None


In [3]:
# merge the chunks to get the merged parquet


## 2) Inspect AlphaEarth Embeddings DuckDB

Verify the table schema, row count, and that `sample_id` and the 64 embedding columns (`A00`–`A63`) are present.

In [4]:

# ==============================================================
# 2) Inspect embeddings parquet
# ==============================================================
print(f"Loading {COG_EMBEDDINGS_PARQUET.name} …")
df_snap = pd.read_parquet(str(COG_EMBEDDINGS_PARQUET))

print(f"\nTotal rows      : {len(df_snap):,}")
print(f"Valid embeddings: {df_snap['valid'].sum():,}  ({100*df_snap['valid'].mean():.1f}%)")
print(f"Nodata / invalid: {(~df_snap['valid']).sum():,}")

# Identify 64 embedding columns (A00 … A63)
AE_EMBED_COLS = sorted(
    [c for c in df_snap.columns if len(c) == 3 and c[0] == "A" and c[1:].isdigit()],
    key=lambda c: int(c[1:])
)
print(f"\nEmbedding columns ({len(AE_EMBED_COLS)}): {AE_EMBED_COLS[:4]} … {AE_EMBED_COLS[-4:]}")
assert len(AE_EMBED_COLS) == 64, f"Expected 64 embedding cols, got {len(AE_EMBED_COLS)}"
assert "sample_id" in df_snap.columns
assert "lat" in df_snap.columns and "lon" in df_snap.columns

print("\nYear distribution:")
print(df_snap["year"].value_counts().sort_index().to_string())

print("\nSample rows (valid=True):")
display(df_snap[df_snap["valid"]][["sample_id", "lat", "lon", "year", "A00", "A01"]].head(5))

valid = df_snap[df_snap["valid"]]
print(f"\nA00 range : [{valid['A00'].min():.4f},  {valid['A00'].max():.4f}]  (expected [-1, 1])")
print(f"lat range : [{df_snap['lat'].min():.3f},  {df_snap['lat'].max():.3f}]")
print(f"lon range : [{df_snap['lon'].min():.3f},  {df_snap['lon'].max():.3f}]")


Loading alphaearth_cog_embeddings_eastern_africa.parquet …

Total rows      : 115,531
Valid embeddings: 115,107  (99.6%)
Nodata / invalid: 424

Embedding columns (64): ['A00', 'A01', 'A02', 'A03'] … ['A60', 'A61', 'A62', 'A63']

Year distribution:
year
2017     2533
2018    18248
2019    10812
2020    10725
2021    29186
2022    11512
2023    14471
2024     8997
2025     9047

Sample rows (valid=True):


,sample_id,lat,lon,year,A00,A01
0,2018_GLO_GLANCE_POINT_100_03_4_00_2_904796_199...,-12.758500,33.479488,2018,0.094488,0.417323
1,2018_GLO_GLANCE_POINT_100_03_4_00_2_904816_199...,-13.005869,33.343521,2018,0.299213,0.377953
2,2018_GLO_GLANCE_POINT_100_03_4_00_2_905724_199...,-12.727608,33.503973,2018,0.259843,0.425197
3,2018_MWI_WFP-field-survey_POLY_110_0,-12.966103,33.741682,2018,0.275591,0.433071
4,2018_MWI_WFP-field-survey_POLY_110_1,-12.966203,33.742747,2018,0.330709,0.354331



A00 range : [-0.5354,  0.6063]  (expected [-1, 1])
lat range : [-26.813,  17.813]
lon range : [22.016,  57.813]


## 2b) Deduplicate DuckDB

The `raw` table may contain multiple rows per `sample_id` (e.g. from repeated GEE exports).  This cell:
1. Reports total vs unique `sample_id` counts and shows the worst offenders.
2. Rewrites the table in-place keeping only **one row per `sample_id`** (NULL rows are dropped).
3. Runs a DuckDB `CHECKPOINT` so the compacted file is persisted to disk.

In [5]:

# ==============================================================
# 2b) Deduplicate (parquet was deduplicated during merge, but verify)
# ==============================================================
n_before  = len(df_snap)
null_ids  = int(df_snap["sample_id"].isna().sum())
df_snap   = df_snap.dropna(subset=["sample_id"]).drop_duplicates(subset="sample_id").copy()
removed   = n_before - len(df_snap)

print(f"Before : {n_before:,}  (null sample_ids: {null_ids:,})")
print(f"After  : {len(df_snap):,}  (removed {removed:,} duplicate/null rows)")

if removed > 0:
    df_snap.to_parquet(str(COG_EMBEDDINGS_PARQUET), index=False)
    print(f"✓ Saved deduplicated parquet → {COG_EMBEDDINGS_PARQUET.name}")
else:
    print("No duplicates — parquet is clean.")


Before : 115,531  (null sample_ids: 0)
After  : 115,531  (removed 0 duplicate/null rows)
No duplicates — parquet is clean.


## 2c) Regional Coverage — Global Parquet Cross-check

Cross-references the DuckDB `sample_id`s against the full global wide parquet to answer:
- **Which regions** have samples in the DuckDB (and what fraction of each region is covered)?
- **Which `ref_id`s are missing** — these need to be resubmitted to Google Earth Engine to get AlphaEarth embeddings.

Global parquet: `/projects/worldcereal/data/cached_wide_merged/worldcereal_all_extractions_wide_month_with_anomalies_new.parquet`

In [6]:

GLOBAL_PARQUET = Path(
    "/projects/worldcereal/data/cached_wide_merged/"
    "worldcereal_all_extractions_wide_month_with_anomalies_new.parquet"
)
assert GLOBAL_PARQUET.exists(), f"Global parquet not found: {GLOBAL_PARQUET}"

print(f"Reading sample_id / ref_id / region from global parquet …")
df_global = pd.read_parquet(str(GLOBAL_PARQUET), columns=["sample_id", "ref_id", "region"])
df_global.drop_duplicates(subset="sample_id", inplace=True)
df_global.reset_index(drop=True, inplace=True)
print(f"Global parquet unique sample_ids: {len(df_global):,}")
print(f"\nAll regions in global parquet:")
print(df_global["region"].value_counts().to_string())

# ── IDs in the fetched parquet (all rows, valid or not) ──────────────
db_ids = set(df_snap["sample_id"].dropna().unique())
print(f"\nFetched parquet unique sample_ids: {len(db_ids):,}")

# ── Coverage by region ────────────────────────────────────────────────
df_global["in_duckdb"] = df_global["sample_id"].isin(db_ids)

coverage = (
    df_global.groupby("region")
    .agg(total=("sample_id", "count"), in_duckdb=("in_duckdb", "sum"))
    .assign(coverage_pct=lambda x: (100 * x["in_duckdb"] / x["total"]).round(1))
    .sort_values("in_duckdb", ascending=False)
)
print("\n=== Fetched parquet coverage per region ===")
print(coverage[coverage["in_duckdb"] > 0].to_string())


Reading sample_id / ref_id / region from global parquet …
Global parquet unique sample_ids: 7,345,983

All regions in global parquet:
region
Western Europe               3025477
Northern Europe              1369474
Eastern Europe                910241
Southern Europe               636629
South America                 430636
Northern America              284829
Eastern Asia                  174309
Eastern Africa                115531
South-Eastern Asia             76614
Southern Asia                  75841
Western Africa                 53232
Northern Asia                  36054
Central America                32716
Australia and New Zealand      32136
Middle Africa                  28936
Northern Africa                19973
Central Asia                   15613
Southern Africa                14105
Western Asia                    9899
Melanesia                       2011
Caribbean                       1684
Micronesia                        30
Polynesia                          8

Fetched

In [7]:
# ── Eastern Africa: missing ref_ids → resubmit to GEE ─────────────────
sa_df      = df_global[df_global["region"] == "Eastern Africa"].copy()
sa_missing = sa_df[~sa_df["in_duckdb"]].copy()

print(f"=== Eastern Africa ===")
print(f"  Total sample_ids : {len(sa_df):,}")
print(f"  In DuckDB        : {int(sa_df['in_duckdb'].sum()):,}  "
      f"({100*sa_df['in_duckdb'].mean():.1f}%)")
print(f"  Missing          : {len(sa_missing):,}  "
      f"({100*len(sa_missing)/len(sa_df):.1f}%)")

print("\nMissing sample counts per ref_id:")
miss_by_ref = (
    sa_missing.groupby("ref_id")
    .size()
    .reset_index(name="missing_samples")
    .sort_values("missing_samples", ascending=False)
)
print(miss_by_ref.to_string(index=False))

# ── Save the missing ref_id list for GEE resubmission ─────────────────
missing_ref_ids_path = _OUT_DIR / f"missing_ref_ids_{REGION}_for_GEE.csv"
miss_by_ref.to_csv(str(missing_ref_ids_path), index=False)
print(f"\nRef_id list saved to: {missing_ref_ids_path}")

# Also save the full list of missing sample_ids (useful for targeted GEE exports)
missing_samples_path = _OUT_DIR / f"missing_sample_ids_{REGION}_for_GEE.csv"
sa_missing[["ref_id", "sample_id"]].to_csv(str(missing_samples_path), index=False)
print(f"Sample_id list saved to: {missing_samples_path}")

# ── Quick sanity check: DuckDB sample_ids that are NOT in the global parquet ─
global_ids = set(df_global["sample_id"])
db_only = db_ids - global_ids
if db_only:
    print(f"\n⚠️  {len(db_only):,} sample_ids in DuckDB are NOT in the global parquet!")
    print("   First 10:", list(db_only)[:10])
else:
    print(f"\n✓ All {len(db_ids):,} DuckDB sample_ids are present in the global parquet.")


=== Eastern Africa ===
  Total sample_ids : 115,531
  In DuckDB        : 115,531  (100.0%)
  Missing          : 0  (0.0%)

Missing sample counts per ref_id:
Empty DataFrame
Columns: [ref_id, missing_samples]
Index: []

Ref_id list saved to: /path/to/TestFolder/wc_outliers/data_for_outlier/alphaearth_scores_Eastern_Africa/missing_ref_ids_Eastern_Africa_for_GEE.csv
Sample_id list saved to: /path/to/TestFolder/wc_outliers/data_for_outlier/alphaearth_scores_Eastern_Africa/missing_sample_ids_Eastern_Africa_for_GEE.csv

✓ All 115,531 DuckDB sample_ids are present in the global parquet.


## 3) Load & Clean Parquet

Read the parquet, drop existing anomaly columns, and verify sample_id overlap with the embeddings DuckDB.

In [8]:

df_region = pd.read_parquet(str(INPUT_PARQUET))
print(f"Shape (before cleaning): {df_region.shape}")

# ── Drop existing anomaly columns if present ──────────────────────────
cols_to_drop = [c for c in ANOMALY_COLS if c in df_region.columns]
if cols_to_drop:
    df_region.drop(columns=cols_to_drop, inplace=True)
    print(f"Dropped existing anomaly columns: {cols_to_drop}")
else:
    print("No existing anomaly columns to drop.")
print(f"Shape (after cleaning) : {df_region.shape}")

for col in ["ref_id", "sample_id", "ewoc_code", "h3_l3_cell"]:
    if col in df_region.columns:
        print(f"  {col}: {df_region[col].nunique():,} unique values")
    else:
        print(f"  WARNING: '{col}' NOT in parquet!")

print(f"\nUnique sample_ids in parquet : {df_region['sample_id'].nunique():,}")

# ── Overlap with fetched embeddings ──────────────────────────────────
parquet_ids = set(df_region["sample_id"].dropna().unique())
overlap = parquet_ids & db_ids
print(f"\nRegion parquet sample_ids  : {len(parquet_ids):,}")
print(f"Fetched embeddings ids     : {len(db_ids):,}")
print(f"Overlap                    : {len(overlap):,} ({100*len(overlap)/max(len(parquet_ids),1):.1f}% of parquet)")


Shape (before cleaning): (115531, 282)
Dropped existing anomaly columns: ['CTY24_confidence_nonoutlier', 'CTY24_anomaly_flag', 'outlier_CTY24_cls', 'LC10_confidence_nonoutlier', 'LC10_anomaly_flag', 'outlier_LC10_cls']
Shape (after cleaning) : (115531, 276)
  ref_id: 75 unique values
  sample_id: 115,531 unique values
  ewoc_code: 286 unique values
  h3_l3_cell: 589 unique values

Unique sample_ids in parquet : 115,531

Region parquet sample_ids  : 115,531
Fetched embeddings ids     : 115,531
Overlap                    : 115,531 (100.0% of parquet)


## 4) Prepare Embeddings DataFrame

Load the AlphaEarth embeddings, rename `A00`–`A63` → `embedding_0`–`embedding_63`, add a synthetic `model_hash`, and join with parquet metadata (`ref_id`, `ewoc_code`, `h3_l3_cell`) on `sample_id`.

If `h3_l3_cell` is absent from the DuckDB table it is computed from `lat`/`lon` using the `h3` library at resolution 3.

In [9]:

# ── Load valid embeddings from in-memory snapshot ─────────────────────
print(f"Source : {COG_EMBEDDINGS_PARQUET.name}  ({len(df_snap):,} total rows in df_snap)")

load_cols = ["sample_id", "lat", "lon"] + AE_EMBED_COLS
df_emb = df_snap[df_snap["valid"] == True][load_cols].copy().reset_index(drop=True)
print(f"Valid embeddings to score  : {len(df_emb):,}")

# ── Rename A00…A63 → embedding_0…embedding_63 ────────────────────────
rename_map = {ae: f"embedding_{i}" for i, ae in enumerate(AE_EMBED_COLS)}
df_emb.rename(columns=rename_map, inplace=True)
embed_cols = [f"embedding_{i}" for i in range(len(AE_EMBED_COLS))]
print(f"Renamed {len(embed_cols)} embedding columns: {embed_cols[:3]} … {embed_cols[-3:]}")

# ── Cast embeddings to float32 ────────────────────────────────────────
df_emb[embed_cols] = df_emb[embed_cols].astype("float32")

# ── Add synthetic model_hash ──────────────────────────────────────────
df_emb["model_hash"] = MODEL_HASH_PLACEHOLDER

# ── Build parquet metadata lookup: sample_id → ref_id, ewoc_code, h3_l3_cell ──
meta_cols = ["sample_id", "ref_id", "ewoc_code"]
if "h3_l3_cell" in df_region.columns:
    meta_cols.append("h3_l3_cell")
if "lat" in df_region.columns and "lon" in df_region.columns:
    meta_cols += ["lat", "lon"]

df_meta = df_region[meta_cols].drop_duplicates(subset="sample_id").copy()
print(f"Metadata rows (unique sample_id): {len(df_meta):,}")

# ── Left-join embeddings with metadata ───────────────────────────────
df_emb = df_emb.merge(df_meta, on="sample_id", how="left", suffixes=("", "_meta"))

for coord in ("lat", "lon"):
    meta_col = f"{coord}_meta"
    if meta_col in df_emb.columns:
        df_emb[coord] = df_emb[coord].combine_first(df_emb[meta_col])
        df_emb.drop(columns=[meta_col], inplace=True)

print(f"After join: {len(df_emb):,} rows")
print(f"  Null ref_id   : {df_emb['ref_id'].isna().sum():,}")
print(f"  Null ewoc_code: {df_emb['ewoc_code'].isna().sum():,}")

# ── Compute h3_l3_cell if not already present ─────────────────────────
# Replace empty strings → None so h3.cell_to_parent won't raise ValueError
if "h3_l3_cell" in df_emb.columns:
    n_empty = int((df_emb["h3_l3_cell"] == "").sum())
    if n_empty:
        print(f"  Replacing {n_empty:,} empty-string h3_l3_cell values with None …")
    df_emb["h3_l3_cell"] = df_emb["h3_l3_cell"].replace("", None)

needs_h3 = "h3_l3_cell" not in df_emb.columns or df_emb["h3_l3_cell"].isna().any()
if needs_h3:
    n_missing = df_emb["h3_l3_cell"].isna().sum() if "h3_l3_cell" in df_emb.columns else len(df_emb)
    print(f"Computing h3_l3_cell for {n_missing:,} rows from lat/lon at resolution 3 …")
    try:
        import h3
        if "h3_l3_cell" not in df_emb.columns:
            df_emb["h3_l3_cell"] = None
        null_mask = df_emb["h3_l3_cell"].isna()
        df_emb.loc[null_mask, "h3_l3_cell"] = df_emb[null_mask].apply(
            lambda r: h3.latlng_to_cell(r["lat"], r["lon"], 3)
            if pd.notna(r["lat"]) and pd.notna(r["lon"]) else None,
            axis=1,
        )
        print(f"  h3_l3_cell computed — {df_emb['h3_l3_cell'].notna().sum():,} non-null")
    except ImportError:
        print("  WARNING: h3 library not installed — h3_l3_cell will remain null.")
        df_emb["h3_l3_cell"] = None
else:
    print(f"h3_l3_cell present — {df_emb['h3_l3_cell'].notna().sum():,} non-null")

df_emb["ewoc_code"] = df_emb["ewoc_code"].astype(str)

# ── Drop rows with no parquet metadata match ──────────────────────────
df_emb = df_emb[df_emb["ref_id"].notna()].copy()
print(f"\nFinal embeddings DataFrame: {df_emb.shape}")
display(df_emb[["sample_id", "ref_id", "ewoc_code", "h3_l3_cell", "lat", "lon",
                 embed_cols[0], embed_cols[1]]].head())


Source : alphaearth_cog_embeddings_eastern_africa.parquet  (115,531 total rows in df_snap)
Valid embeddings to score  : 115,107
Renamed 64 embedding columns: ['embedding_0', 'embedding_1', 'embedding_2'] … ['embedding_61', 'embedding_62', 'embedding_63']
Metadata rows (unique sample_id): 115,531
After join: 115,107 rows
  Null ref_id   : 0
  Null ewoc_code: 0
  Replacing 32 empty-string h3_l3_cell values with None …
Computing h3_l3_cell for 32 rows from lat/lon at resolution 3 …
  h3_l3_cell computed — 115,107 non-null

Final embeddings DataFrame: (115107, 71)


,sample_id,ref_id,ewoc_code,h3_l3_cell,lat,lon,embedding_0,embedding_1
0,2018_GLO_GLANCE_POINT_100_03_4_00_2_904796_199...,2018_GLO_GLANCE_POINT_100,2501000000,839789fffffffff,-12.758500,33.479488,0.094488,0.417323
1,2018_GLO_GLANCE_POINT_100_03_4_00_2_904816_199...,2018_GLO_GLANCE_POINT_100,2501000000,839789fffffffff,-13.005869,33.343521,0.299213,0.377953
2,2018_GLO_GLANCE_POINT_100_03_4_00_2_905724_199...,2018_GLO_GLANCE_POINT_100,1100000000,839789fffffffff,-12.727608,33.503973,0.259843,0.425197
3,2018_MWI_WFP-field-survey_POLY_110_0,2018_MWI_WFP-field-survey_POLY_110,1101060000,83978dfffffffff,-12.966103,33.741682,0.275591,0.433071
4,2018_MWI_WFP-field-survey_POLY_110_1,2018_MWI_WFP-field-survey_POLY_110,1101060000,83978dfffffffff,-12.966203,33.742747,0.330709,0.354331


## 5) Load Class Mappings

Tries a local `class_mappings.json` first (found automatically in the active conda env's `worldcereal` package).  Falls back to fetching from SharePoint if the local file is not found.

In [10]:
if LOCAL_CLASS_MAPPINGS_JSON is not None and LOCAL_CLASS_MAPPINGS_JSON.exists():
    print(f"Loading class mappings from local file:\n  {LOCAL_CLASS_MAPPINGS_JSON}")
    with open(LOCAL_CLASS_MAPPINGS_JSON) as _f:
        CLASS_MAPPINGS = json.load(_f)
    print("CLASS_MAPPINGS keys:", list(CLASS_MAPPINGS.keys()))

else:
    print("Local class_mappings.json not found — fetching from SharePoint …")
    from dotenv import load_dotenv
    from worldcereal.utils.sharepoint import get_excel_from_sharepoint, build_class_mappings

    _env_path = next((p for p in SHAREPOINT_ENV_CANDIDATES if p.exists()), None)
    assert _env_path is not None, (
        f".sharepointenv not found at: {[str(p) for p in SHAREPOINT_ENV_CANDIDATES]}"
    )
    print(f"Using .sharepointenv: {_env_path}")
    load_dotenv(_env_path, override=True)

    legend = get_excel_from_sharepoint(
        site_url=os.environ["WORLDCEREAL_SP_SITE_URL"],
        file_server_relative_url=os.environ["WORLDCEREAL_SP_FILE_URL"],
        retries=10,
        sheet_name=0,
    )
    legend["ewoc_code"] = (
        legend["ewoc_code"]
        .astype("string")
        .str.replace("-", "", regex=False)
        .pipe(pd.to_numeric, errors="coerce")
        .astype("Int64")
    )
    CLASS_MAPPINGS = build_class_mappings(legend)
    print("CLASS_MAPPINGS keys:", list(CLASS_MAPPINGS.keys()))

assert "LANDCOVER10" in CLASS_MAPPINGS, "LANDCOVER10 key missing from CLASS_MAPPINGS!"
assert "CROPTYPE24"  in CLASS_MAPPINGS, "CROPTYPE24 key missing from CLASS_MAPPINGS!"
print("\nClass mappings loaded successfully.")


Local class_mappings.json not found — fetching from SharePoint …
Using .sharepointenv: /path/to/TestFolder/.sharepointenv
CLASS_MAPPINGS keys: ['LANDCOVER10', 'CROPTYPE24']

Class mappings loaded successfully.


## 6) Scoring — LANDCOVER10

Runs the outlier pipeline on AlphaEarth embeddings for the `LANDCOVER10` label schema by passing `embeddings_df=(df_emb, embed_cols)`.  This bypasses the DuckDB reads inside `run_pipeline` entirely.

In [11]:
LC10_flagged_gdf, LC10_summary_df = run_pipeline(
    embeddings_db_path=None,          # not used — embeddings_df takes precedence
    restrict_model_hash=None,
    label_domain="LANDCOVER10",
    map_to_finetune=False,
    class_mappings_name="LANDCOVER10",
    skip_classes=SKIP_CLASSES,
    mapping_file=CLASS_MAPPINGS,
    h3_level=LC10_H3_LEVELS,
    # -- Slice definition ---------------------------------------------
    # group_cols joins the slice key on top of the H3 cell and the label.
    #   ["ref_id"]  (DEFAULT)  score each source dataset against itself, so one
    #               dataset's labelling convention cannot contaminate another's
    #               reference cloud.
    #   []                     pool across datasets (the previous behaviour).
    #               Pooling lets a dataset digitised against a different legend
    #               either mask itself (if it is large) or be flagged wholesale
    #               (if it is small).
    # NOTE the *context* for the purity / margin signals stays geographic either
    # way -- it deliberately does not inherit group_cols, or single-crop datasets
    # would lose those votes entirely.  Override with context_group_cols.
    group_cols=["ref_id"],
    min_slice_size=LC10_MIN_SLICE,
    max_slice_size=LC10_MAX_SLICE,
    merge_small_slice=True,
    max_merge_iterations=LC10_MAX_MERGE,
    threshold_mode="stable_mad",   # local median, cross-slice sigma
    percentile_q=0.96,
    mad_k=MAD_K,
    abs_threshold=None,
    fdr_alpha=0.05,
    min_flagged_per_slice=None,
    max_flagged_fraction=None,
    max_full_pairwise_n=0,
    norm_percentiles=NORM_PERCENTILES,
    # -- Absolute gate (see README, "How a sample gets flagged") ------
    # A flag now needs BOTH a within-slice signal and an absolute one, measured
    # against a null pooled across many slices of the same class.  Without it the
    # top ~2% of EVERY slice clears the escalation thresholds whether or not the
    # slice holds a single real error.  Set require_absolute=False to reproduce
    # the old relative-only behaviour for an ablation.
    require_absolute=True,
    abs_z_k=3.3,                # robust sigma required to flag
    abs_z_suspect=4.0,
    abs_z_candidate=5.5,
    abs_combine="min",          # demand centroid AND neighbourhood evidence
    # -- Heavy slice contamination ------------------------------------
    # The scale, not the centroid, was the blocker: the cross-slice null took
    # the median of per-slice MADs, and a MAD is widened by the very right-side
    # errors it measures against -- so when EVERY slice of a class carries
    # 30-45% errors the null inflated with them and nothing cleared the gate.
    # "left_tail" measures the spread as median-q25, i.e. from the clean left
    # half only. Measured recall at 40% slice contamination: 0.14 -> 0.90, at a
    # slightly LOWER clean false-positive rate. Use "mad" for ablations only.
    null_scale_estimator="left_tail",
    purity_veto=0.80,           # cap escalation when the neighbours agree
    # -- Coverage / quality -------------------------------------------
    min_scoring_slice_size=50,  # below this -> flag "unscored", not "normal"
    quality_gate=True,          # quarantine degenerate embeddings as "unscorable"
    strict_quality=False,       # True = hard-error on mixed model_hash / bad H3
    # -- Temporal control ---------------------------------------------
    # Embeddings are season-specific and the collection spans many years, so a
    # sample from a minority year is distant for phenological, not label, reasons.
    # Point this at a year/season column IF the embeddings carry one -- the
    # standard cache schema does not, and the run will say so if it cannot.
    time_col=None,
    output_samples_path=LC10_SAMPLES_PATH,
    output_summary_path=LC10_SUMMARY_PATH,
    debug=False,
    centroid_mode=CENTROID_MODE,
    centroid_trim=CENTROID_TRIM,
    gate_confidence_by_flag=True,
    apply_slice_trust=False,
    slice_trust_min=0.05,
    embeddings_df=(df_emb, embed_cols),   # ← inject AlphaEarth embeddings directly
)
print(f"LC10 pipeline done — {len(LC10_flagged_gdf):,} samples scored.")


[anomaly] Using pre-supplied embeddings: 115,107 rows
[anomaly] Loaded 115,107 rows from embeddings_cache
[anomaly] Mapping classes using mapping_file
[anomaly] skip_classes ['ignore', 'no_crop', 'trees']: held aside 16,387 rows, processing 98,720 rows.
[anomaly] Preparing embeddings array...
[anomaly] count_before_drop: 98,720
[anomaly] count_after_drop: 98,720
[anomaly] Dropped 0 rows with missing label columns ['LANDCOVER10'] and dropped!
[anomaly] Adaptive H3 mode: levels [1, 2] (finest→coarsest), min_slice_size=200
[anomaly] Max slice size cap: 10,000
[adaptive_h3]   L1: 173 slices resolved (76,575 pts), 1 slices too big (22,145 pts) → next level
[adaptive_h3] Level 1: 76,575 points
[adaptive_h3] Level 2: 22,145 points
[anomaly] Merging small slices (min_size=200)... [180 slices before merge]
[anomaly] After merge: 64 slices
[anomaly] Computing context centroid metrics...
[anomaly] Scoring slices...
[anomaly] Computing per-slice centroids...


Scoring slices: 100%|██████████| 64/64 [00:28<00:00,  2.27slice/s, 8,882 pts | temporary_crops]      


[anomaly] Flagging anomalies (mode=mad)...
[anomaly] Computing robust confidence for flagged points...
[anomaly] Computing kNN label purity for flagged points...
[anomaly] Applying confidence fusion...


/path/to/.conda/envs/radix_update/lib/python3.10/site-packages/EBA_detector/anomaly.py:965: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  flagged_df = pd.concat([flagged_df, df_skipped], axis=0, ignore_index=True)


[anomaly] Re-attached 16,387 skipped-class rows with NaN scores.
[anomaly] Writing flagged samples -> /path/to/TestFolder/wc_outliers/data_for_outlier/alphaearth_scores_Eastern_Africa/LC10/LC10_samples_Eastern_Africa.parquet
[anomaly] Writing summary -> /path/to/TestFolder/wc_outliers/data_for_outlier/alphaearth_scores_Eastern_Africa/LC10/LC10_summary_Eastern_Africa.parquet
LC10 pipeline done — 115,107 samples scored.


In [12]:
# ── Read back & rename for downstream merging ─────────────────────────
# (also useful for resuming the notebook without re-running run_pipeline)
_lc10_cols = ["ref_id", "sample_id", "LANDCOVER10", "confidence_nonoutlier", "anomaly_flag"]
LC10_flagged_gdf = pd.read_parquet(LC10_SAMPLES_PATH, columns=_lc10_cols)

LC10_flagged_gdf = LC10_flagged_gdf.rename(columns={
    "confidence_nonoutlier": "LC10_confidence_nonoutlier",
    "anomaly_flag":          "LC10_anomaly_flag",
    "LANDCOVER10":           "outlier_LC10_cls",
})
print(f"LC10 scores: {len(LC10_flagged_gdf):,} rows")
print(LC10_flagged_gdf["LC10_anomaly_flag"].value_counts())
display(LC10_flagged_gdf.head())


LC10 scores: 115,107 rows
LC10_anomaly_flag
normal       91503
flagged       5867
suspect        839
candidate      511
Name: count, dtype: int64


,ref_id,sample_id,outlier_LC10_cls,LC10_confidence_nonoutlier,LC10_anomaly_flag
0,2019_GLO_EWOCO-v2_POINT_100,2019_GLO_EWOCO-v2_POINT_100_7884,bare_sparsely_vegetated,0.010000,candidate
1,2018_GLO_EWOCO-v2_POINT_100,2018_GLO_EWOCO-v2_POINT_100_38929,bare_sparsely_vegetated,0.863682,suspect
2,2020_GLO_EWOCO-v2_POINT_100,2020_GLO_EWOCO-v2_POINT_100_7852,bare_sparsely_vegetated,0.849917,suspect
3,2018_GLO_EWOCO-v2_POINT_100,2018_GLO_EWOCO-v2_POINT_100_59071,bare_sparsely_vegetated,0.989826,flagged
4,2020_GLO_EWOCO-v2_POINT_100,2020_GLO_EWOCO-v2_POINT_100_42841,bare_sparsely_vegetated,1.000000,flagged


## 7) Scoring — CROPTYPE24

Same approach with the finer `CROPTYPE24` schema: three H3 levels and a smaller per-slice cap.

In [13]:
CTY24_flagged_gdf, CTY24_summary_df = run_pipeline(
    embeddings_db_path=None,          # not used — embeddings_df takes precedence
    restrict_model_hash=None,
    label_domain="CROPTYPE24",
    map_to_finetune=False,
    class_mappings_name="CROPTYPE24",
    skip_classes=SKIP_CLASSES,
    mapping_file=CLASS_MAPPINGS,
    h3_level=CTY24_H3_LEVELS,
    # -- Slice definition ---------------------------------------------
    # group_cols joins the slice key on top of the H3 cell and the label.
    #   ["ref_id"]  (DEFAULT)  score each source dataset against itself, so one
    #               dataset's labelling convention cannot contaminate another's
    #               reference cloud.
    #   []                     pool across datasets (the previous behaviour).
    #               Pooling lets a dataset digitised against a different legend
    #               either mask itself (if it is large) or be flagged wholesale
    #               (if it is small).
    # NOTE the *context* for the purity / margin signals stays geographic either
    # way -- it deliberately does not inherit group_cols, or single-crop datasets
    # would lose those votes entirely.  Override with context_group_cols.
    group_cols=["ref_id"],
    min_slice_size=CTY24_MIN_SLICE,
    max_slice_size=CTY24_MAX_SLICE,
    merge_small_slice=True,
    max_merge_iterations=CTY24_MAX_MERGE,
    threshold_mode="stable_mad",   # local median, cross-slice sigma
    percentile_q=0.96,
    mad_k=MAD_K,
    abs_threshold=None,
    fdr_alpha=0.05,
    min_flagged_per_slice=None,
    max_flagged_fraction=None,
    max_full_pairwise_n=0,
    norm_percentiles=NORM_PERCENTILES,
    # -- Absolute gate (see README, "How a sample gets flagged") ------
    # A flag now needs BOTH a within-slice signal and an absolute one, measured
    # against a null pooled across many slices of the same class.  Without it the
    # top ~2% of EVERY slice clears the escalation thresholds whether or not the
    # slice holds a single real error.  Set require_absolute=False to reproduce
    # the old relative-only behaviour for an ablation.
    require_absolute=True,
    abs_z_k=3.3,                # robust sigma required to flag
    abs_z_suspect=4.0,
    abs_z_candidate=5.5,
    abs_combine="min",          # demand centroid AND neighbourhood evidence
    # -- Heavy slice contamination ------------------------------------
    # The scale, not the centroid, was the blocker: the cross-slice null took
    # the median of per-slice MADs, and a MAD is widened by the very right-side
    # errors it measures against -- so when EVERY slice of a class carries
    # 30-45% errors the null inflated with them and nothing cleared the gate.
    # "left_tail" measures the spread as median-q25, i.e. from the clean left
    # half only. Measured recall at 40% slice contamination: 0.14 -> 0.90, at a
    # slightly LOWER clean false-positive rate. Use "mad" for ablations only.
    null_scale_estimator="left_tail",
    purity_veto=0.80,           # cap escalation when the neighbours agree
    # -- Coverage / quality -------------------------------------------
    min_scoring_slice_size=50,  # below this -> flag "unscored", not "normal"
    quality_gate=True,          # quarantine degenerate embeddings as "unscorable"
    strict_quality=False,       # True = hard-error on mixed model_hash / bad H3
    # -- Temporal control ---------------------------------------------
    # Embeddings are season-specific and the collection spans many years, so a
    # sample from a minority year is distant for phenological, not label, reasons.
    # Point this at a year/season column IF the embeddings carry one -- the
    # standard cache schema does not, and the run will say so if it cannot.
    time_col=None,
    output_samples_path=CTY24_SAMPLES_PATH,
    output_summary_path=CTY24_SUMMARY_PATH,
    debug=False,
    centroid_mode=CENTROID_MODE,
    centroid_trim=CENTROID_TRIM,
    gate_confidence_by_flag=True,
    apply_slice_trust=False,
    slice_trust_min=0.05,
    embeddings_df=(df_emb, embed_cols),   # ← inject AlphaEarth embeddings directly
)
print(f"CTY24 pipeline done — {len(CTY24_flagged_gdf):,} samples scored.")


[anomaly] Using pre-supplied embeddings: 115,107 rows
[anomaly] Loaded 115,107 rows from embeddings_cache
[anomaly] Mapping classes using mapping_file
[anomaly] skip_classes ['ignore', 'no_crop', 'trees']: held aside 68,471 rows, processing 46,636 rows.
[anomaly] Preparing embeddings array...
[anomaly] count_before_drop: 46,636
[anomaly] count_after_drop: 46,636
[anomaly] Dropped 0 rows with missing label columns ['CROPTYPE24'] and dropped!
[anomaly] Adaptive H3 mode: levels [1, 2, 3] (finest→coarsest), min_slice_size=100
[anomaly] Max slice size cap: 5,000
[adaptive_h3]   L1: 158 slices resolved (33,699 pts), 1 slices too big (12,937 pts) → next level
[adaptive_h3]   L2: 2 slices resolved (1,251 pts), 2 slices too big (11,686 pts) → next level
[adaptive_h3] Level 1: 33,699 points
[adaptive_h3] Level 2: 1,251 points
[adaptive_h3] Level 3: 11,686 points
[anomaly] Merging small slices (min_size=100)... [173 slices before merge]
[anomaly] After merge: 92 slices
[anomaly] Computing context

Scoring slices: 100%|██████████| 92/92 [00:11<00:00,  7.73slice/s, 2,755 pts | maize]           


[anomaly] Flagging anomalies (mode=mad)...
[anomaly] Computing robust confidence for flagged points...
[anomaly] Computing kNN label purity for flagged points...
[anomaly] Applying confidence fusion...
[anomaly] Re-attached 68,471 skipped-class rows with NaN scores.


/path/to/.conda/envs/radix_update/lib/python3.10/site-packages/EBA_detector/anomaly.py:965: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  flagged_df = pd.concat([flagged_df, df_skipped], axis=0, ignore_index=True)


[anomaly] Writing flagged samples -> /path/to/TestFolder/wc_outliers/data_for_outlier/alphaearth_scores_Eastern_Africa/CTY24/CTY24_samples_Eastern_Africa.parquet
[anomaly] Writing summary -> /path/to/TestFolder/wc_outliers/data_for_outlier/alphaearth_scores_Eastern_Africa/CTY24/CTY24_summary_Eastern_Africa.parquet
CTY24 pipeline done — 115,107 samples scored.


In [14]:
# ── Read back & rename for downstream merging ─────────────────────────
_cty24_cols = ["ref_id", "sample_id", "CROPTYPE24", "confidence_nonoutlier", "anomaly_flag"]
CTY24_flagged_gdf = pd.read_parquet(CTY24_SAMPLES_PATH, columns=_cty24_cols)

CTY24_flagged_gdf = CTY24_flagged_gdf.rename(columns={
    "confidence_nonoutlier": "CTY24_confidence_nonoutlier",
    "anomaly_flag":          "CTY24_anomaly_flag",
    "CROPTYPE24":            "outlier_CTY24_cls",
})
print(f"CTY24 scores: {len(CTY24_flagged_gdf):,} rows")
print(CTY24_flagged_gdf["CTY24_anomaly_flag"].value_counts())
display(CTY24_flagged_gdf.head())


CTY24 scores: 115,107 rows
CTY24_anomaly_flag
normal       43543
flagged       2439
suspect        395
candidate      259
Name: count, dtype: int64


,ref_id,sample_id,outlier_CTY24_cls,CTY24_confidence_nonoutlier,CTY24_anomaly_flag
0,2020_ETH_EthCT2020_POINT_110,2020_ETH_EthCT2020_POINT_110_1420,wheat,0.721501,suspect
1,2020_ETH_EthCT2020_POINT_110,2020_ETH_EthCT2020_POINT_110_1939,wheat,0.168353,candidate
2,2020_ETH_EthCT2020_POINT_110,2020_ETH_EthCT2020_POINT_110_2405,wheat,0.010000,candidate
3,2020_ETH_EthCT2020_POINT_110,2020_ETH_EthCT2020_POINT_110_1441,wheat,1.000000,flagged
4,2020_ETH_EthCT2020_POINT_110,2020_ETH_EthCT2020_POINT_110_1200,wheat,1.000000,flagged


## 8) Merge LC10 + CTY24 Scores

Outer-join the two score tables on `(ref_id, sample_id)`.  Handles the case where one DataFrame is empty.

In [15]:
if LC10_flagged_gdf.empty and CTY24_flagged_gdf.empty:
    print("Both score DataFrames are empty — nothing to merge.")
    merged_scores = pd.DataFrame(columns=["ref_id", "sample_id"] + ANOMALY_COLS)
elif LC10_flagged_gdf.empty:
    merged_scores = CTY24_flagged_gdf.copy()
    for col in ["LC10_confidence_nonoutlier", "LC10_anomaly_flag", "outlier_LC10_cls"]:
        merged_scores[col] = float("nan") if "confidence" in col else None
elif CTY24_flagged_gdf.empty:
    merged_scores = LC10_flagged_gdf.copy()
    for col in ["CTY24_confidence_nonoutlier", "CTY24_anomaly_flag", "outlier_CTY24_cls"]:
        merged_scores[col] = float("nan") if "confidence" in col else None
else:
    merged_scores = CTY24_flagged_gdf.merge(
        LC10_flagged_gdf, on=["ref_id", "sample_id"], how="outer"
    )

merged_scores.sort_values(["ref_id", "sample_id"], inplace=True)
merged_scores.reset_index(drop=True, inplace=True)

print(f"Merged scores: {len(merged_scores):,} rows × {merged_scores.shape[1]} columns")
print("\nLC10_anomaly_flag:")
print(merged_scores["LC10_anomaly_flag"].value_counts(dropna=False))
print("\nCTY24_anomaly_flag:")
print(merged_scores["CTY24_anomaly_flag"].value_counts(dropna=False))

# ── Save to disk ───────────────────────────────────────────────────────
merged_scores.to_parquet(str(MERGED_SCORES_PATH), index=False)
print(f"\nMerged scores saved to: {MERGED_SCORES_PATH}")

display(merged_scores.head())


Merged scores: 115,107 rows × 8 columns

LC10_anomaly_flag:
LC10_anomaly_flag
normal       91503
None         16387
flagged       5867
suspect        839
candidate      511
Name: count, dtype: int64

CTY24_anomaly_flag:
CTY24_anomaly_flag
None         68471
normal       43543
flagged       2439
suspect        395
candidate      259
Name: count, dtype: int64

Merged scores saved to: /path/to/TestFolder/wc_outliers/data_for_outlier/alphaearth_scores_Eastern_Africa/merged_LC10_CTY24_Eastern_Africa.parquet


,ref_id,sample_id,outlier_CTY24_cls,CTY24_confidence_nonoutlier,CTY24_anomaly_flag,outlier_LC10_cls,LC10_confidence_nonoutlier,LC10_anomaly_flag
0,2017_AF_One-Acre-Fund-MEL_POINT_110,2017_AF_One-Acre-Fund-MEL_POINT_110_2017_AF_OA...,maize,1.0,normal,temporary_crops,1.0,normal
1,2017_AF_One-Acre-Fund-MEL_POINT_110,2017_AF_One-Acre-Fund-MEL_POINT_110_2017_AF_OA...,maize,1.0,normal,temporary_crops,1.0,normal
2,2017_AF_One-Acre-Fund-MEL_POINT_110,2017_AF_One-Acre-Fund-MEL_POINT_110_2017_AF_OA...,maize,1.0,normal,temporary_crops,1.0,normal
3,2017_AF_One-Acre-Fund-MEL_POINT_110,2017_AF_One-Acre-Fund-MEL_POINT_110_2017_AF_OA...,maize,1.0,normal,temporary_crops,1.0,normal
4,2017_AF_One-Acre-Fund-MEL_POINT_110,2017_AF_One-Acre-Fund-MEL_POINT_110_2017_AF_OA...,maize,1.0,flagged,temporary_crops,1.0,flagged


## 9) Write Scores Back to Parquet

Left-join the merged scores onto the cleaned `Eastern_Africa` DataFrame and write to `OUTPUT_PARQUET`.

In [16]:
# ── Read merged scores from disk (safe even if above cells were skipped) ──
merged_scores = pd.read_parquet(str(MERGED_SCORES_PATH))

# ── Build lookup: one row per sample_id ──────────────────────────────
scores_lookup = (
    merged_scores[["ref_id", "sample_id"] + ANOMALY_COLS]
    .drop_duplicates(subset="sample_id")
    .copy()
)
print(f"Scores lookup: {len(scores_lookup):,} unique sample_ids")

# ── Ensure old anomaly columns are gone from df_region ───────────────
cols_to_drop = [c for c in ANOMALY_COLS if c in df_region.columns]
if cols_to_drop:
    df_region.drop(columns=cols_to_drop, inplace=True)

# ── Left-join scores onto the parquet rows ────────────────────────────
df_out = df_region.merge(scores_lookup, on=["ref_id", "sample_id"], how="left")
print(f"Output DataFrame shape: {df_out.shape}")

# ── Join completeness check ───────────────────────────────────────────
for col in ANOMALY_COLS:
    n_null = df_out[col].isna().sum()
    pct = 100 * n_null / max(len(df_out), 1)
    print(f"  {col}: {n_null:,} NaN ({pct:.1f}%)")

# ── Write output ──────────────────────────────────────────────────────
df_out.to_parquet(
    str(OUTPUT_PARQUET),
    index=False,
    compression="zstd",
    row_group_size=20_000,
)
print(f"\nOutput written to: {OUTPUT_PARQUET}")
print(f"Final shape: {df_out.shape}")

print("\n=== LC10_anomaly_flag distribution ===")
print(df_out["LC10_anomaly_flag"].value_counts(dropna=False))
print("\n=== CTY24_anomaly_flag distribution ===")
print(df_out["CTY24_anomaly_flag"].value_counts(dropna=False))


Scores lookup: 115,107 unique sample_ids
Output DataFrame shape: (115531, 282)
  CTY24_confidence_nonoutlier: 68,895 NaN (59.6%)
  CTY24_anomaly_flag: 68,895 NaN (59.6%)
  outlier_CTY24_cls: 424 NaN (0.4%)
  LC10_confidence_nonoutlier: 16,811 NaN (14.6%)
  LC10_anomaly_flag: 16,811 NaN (14.6%)
  outlier_LC10_cls: 424 NaN (0.4%)

Output written to: /path/to/TestFolder/wc_outliers/data_for_outlier/Eastern_Africa_with_alphaearth_scores.parquet
Final shape: (115531, 282)

=== LC10_anomaly_flag distribution ===
LC10_anomaly_flag
normal       91503
None         16387
flagged       5867
suspect        839
candidate      511
NaN            424
Name: count, dtype: int64

=== CTY24_anomaly_flag distribution ===
CTY24_anomaly_flag
None         68471
normal       43543
flagged       2439
NaN            424
suspect        395
candidate      259
Name: count, dtype: int64
